In [1]:
print (11)

11


In [2]:
import sys
print(sys.executable)

/workspaces/llm-zoomcamp-2026-homeworks/llm-zoomcamp-hw2/.venv/bin/python


Load the documents

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Create embeddings

In [4]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(v[0])

2026-06-29 16:55:40.803905714 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


-0.02058203437252893


Chunk the documents

In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

: 

In [ ]:
from gitsource import chunk_documents
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

vectors = embedder.encode_batch(
    [chunk["content"] for chunk in chunks]
)

X = np.vstack(vectors)

scores = X.dot(v)

best_idx = scores.argmax()

print(chunks[best_idx]["filename"])

Q4

In [ ]:
query = "What metric do we use to evaluate a search engine?"

v = embedder.encode(query)

scores = X.dot(v)

best = scores.argmax()

print(chunks[best]["filename"])

04-evaluation/lessons/05-search-metrics.md


Q5

Perform Vector Search (manual)

In [ ]:
query = "How do I store vectors in PostgreSQL?"

v = embedder.encode(query)

scores = X.dot(v)

top5_idx = scores.argsort()[-5:][::-1]

vector_results = [chunks[i] for i in top5_idx]

Display them:

In [ ]:
for doc in vector_results:
    print(doc["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


Build the Text Search Index

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

: 

Index all chunks

In [ ]:
index.fit(chunks)

Run Text Search

In [ ]:
text_results = index.search(
    query="How do I store vectors in PostgreSQL?",
    num_results=5
)

Display them:

In [ ]:
for doc in text_results:
    print(doc["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


Compare both searches

In [ ]:
vector_files = set(doc["filename"] for doc in vector_results)
text_files = set(doc["filename"] for doc in text_results)

print("Vector Search:")
print(vector_files)

print()

print("Text Search:")
print(text_files)

print()

print("Only in Vector Search:")
print(vector_files - text_files)

Vector Search:
{'03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md'}

Text Search:
{'02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md'}

Only in Vector Search:
{'02-vector-search/lessons/08-pgvector.md'}


Q6

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
results = rrf([vector_results, text_results])

: 

In [ ]:
results